# <h1 style="font-family: Trebuchet MS; padding: 12px; font-size: 48px; color:rgba(33, 87, 62, 1); text-align: center; line-height: 1.25;"><b>⚽Pre Match<span style="color: #000000"> Analysis 🎮📉</span></b><br><span style="color:rgb(92, 152, 255); font-size: 24px">with Various Machine Learning Models </span></h1>
<hr>
​

<center><img src="https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExcGZ6czFkZ2hodm9rc3Jta3RiNGcyZmJkcnNtbmFmcHhxcG4xd21pNiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/8kRLZnmWIG95b055FL/giphy.gif"></center>

<a id='top'></a>
<div class="list-group" id="list-tab" role="tablist">
   <p style="background-color:rgba(33, 87, 62, 1); font-family: 'Arial', sans-serif; color:#FFFFFF; font-size:250%; text-align:center; border-radius:15px 15px;">
        Import Libraries📚
    </p>
</div>


In [1]:
api_base = r'http://footballapi.eastus.azurecontainer.io:3000/'

In [2]:
import requests
import pandas as pd
import json
from pandas import DataFrame

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">1. | Getting the id for the last n matches</div>


In [3]:


def get_team_lnm(api_base :str , team_id:int , num_matchs: int) -> dict :
  '''this function takes the base of the api without the endpoint , the team id and the number of last matchs wanted
  and returns a dictionary of the last matchs ids of the team with the home team and away team names'''
  try :
    response = requests.get(api_base +f'teams/{team_id}/events/last/0') # api response
  except:
    return 'the api is down'
  match_info = { # extracting match details
    match['id']: {
        'homeTeam': match['homeTeam']['name'],
        'awayTeam': match['awayTeam']['name']
    }
    for match in response.json()['events']
}
  match_info =  dict(reversed(list(match_info.items())[-num_matchs:])) # filtering the last n matches from the data and reverse them (the last match is the first)
  match_info['target_team_id'] =team_id  # adding the id of the team to the dict
  match_info['target_team_name'] = requests.get(api_base + f'teams/{team_id}').json( ).get('team').get('name') # adding the name of the team to the dict
  return match_info

teams_info = get_team_lnm(api_base , 2829 , 5)
print(teams_info )

{14083112: {'homeTeam': 'Rayo Vallecano', 'awayTeam': 'Real Madrid'}, 14566636: {'homeTeam': 'Liverpool', 'awayTeam': 'Real Madrid'}, 14083099: {'homeTeam': 'Real Madrid', 'awayTeam': 'Valencia'}, 14083729: {'homeTeam': 'Real Madrid', 'awayTeam': 'Barcelona'}, 14566596: {'homeTeam': 'Real Madrid', 'awayTeam': 'Juventus'}, 'target_team_id': 2829, 'target_team_name': 'Real Madrid'}


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">2. | Getting the statistics for the last n matches using id</div>


In [19]:

def get_match_stats(api_base: str , matches_info: dict ) -> DataFrame :
    '''this function takes the base of the api without the endpoint , the match onfo dictionary  and returns the statistics of the matches as a dataframe'''
    all_matches_stats = [] # define a dict to contain all the information to convert it to a DataFrame later
    
    for match_id in matches_info.keys():
        if isinstance(match_id, int):
            # getting the correct key for the values based on whether the team is away or home
            if matches_info.get('target_team_name') == matches_info.get(match_id).get('awayTeam'):
                value_key = 'awayValue'
            else :
                value_key = 'homeValue'
                
            match_stats = {}
            home_team = matches_info[match_id]['homeTeam'] # just adding each match id , home team name and away team name to the dict
            away_team = matches_info[match_id]['awayTeam']
            match_stats['match_id'] = match_id
            match_stats['home_team'] = home_team
            match_stats['away_team'] = away_team
            
            try : # handling the api if it's disconnected or failed
                response = requests.get(api_base + f'events/{match_id}/statistics').json()['statistics'] # getting the statistics of the match
            except :
                return 'api is down'
            
            for period in response: # loop over each period statistics
                if period.get('period') ==  'ALL': # get the general stats only
                    response = period['groups'] # filtering only the overall statistics
                
            for group_stat in response: # loop over each match group statistics
                for stat_item in group_stat['statisticsItems']: # loop over each statistic item (collection of stats under a specific category)
                    if stat_item.get('name') != None :
                        match_stats[stat_item.get('name')] = stat_item.get(value_key) # adding the feature and the value for it in the dict
            
            all_matches_stats.append(match_stats) # adding the whole match stats to the general list

    all_matches_stats = pd.DataFrame(all_matches_stats ).fillna(0) # convert the general list to data frame and fill none values with 0
    return all_matches_stats
        
        
        
matches_stats = get_match_stats(api_base ,teams_info )
matches_stats.head().style.background_gradient(cmap='Greens').set_properties(**{'font-family': 'Segoe UI'})

,match_id,home_team,away_team,Ball possession,Expected goals,Big chances,Total shots,Goalkeeper saves,Corner kicks,Fouls,Passes,Tackles,Free kicks,Yellow cards,Shots on target,Hit woodwork,Shots off target,Blocked shots,Shots inside box,Shots outside box,Big chances missed,Touches in penalty area,Fouled in final third,Offsides,Accurate passes,Throw-ins,Final third entries,Final third phase,Long balls,Crosses,Duels,Dispossessed,Ground duels,Aerial duels,Dribbles,Tackles won,Total tackles,Interceptions,Recoveries,Clearances,Total saves,Goals prevented,Big saves,High claims,Goal kicks,Distance covered,Number of sprints,Big chances scored,Errors lead to a shot,Penalty saves,Red cards,Through balls,Errors lead to a goal,Punches
0,14083112,Rayo Vallecano,Real Madrid,54,0.980000,1,21,2,8,7,445,10,17,2,5,0,10,6,10,11,1,42,5,1,376,26,55,99,23,6,57,9,45,13,18,6,10,10,40,23,2,0.161300,0.000000,1.000000,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,14566636,Liverpool,Real Madrid,61,0.450000,1,8,8,2,11,531,18,16,4,2,0,4,2,3,5,1,25,2,0,461,11,65,137,15,0,55,8,46,10,12,13,18,6,50,29,8,0.759100,3.000000,0.000000,4,112.727760,152.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,14083099,Real Madrid,Valencia,65,2.710000,4,21,1,7,14,715,21,5,1,11,0,4,6,12,9,2,59,3,2,647,22,126,263,33,3,45,10,35,6,8,12,21,7,53,11,1,0.046100,0.000000,0.000000,2,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,14083729,Real Madrid,Barcelona,32,3.630000,4,23,4,12,12,275,24,8,5,10,0,6,7,17,6,2,34,4,5,223,11,43,79,21,4,53,10,45,7,13,13,24,6,53,17,4,-0.362100,0.000000,1.000000,6,0.000000,0.000000,2.000000,1.000000,0.000000,1.000000,2.000000,1.000000,1.000000
4,14566596,Real Madrid,Juventus,66,2.690000,3,28,4,13,10,619,14,18,1,10,1,6,12,17,11,2,39,2,1,560,12,81,212,28,4,59,4,38,12,8,10,14,14,43,22,4,0.285000,2.000000,2.000000,4,105.028570,124.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">3. | Getting the statistics for every player in the match (in match statistics)</div>


In [20]:
def get_players_stats(api_base: str, matches_info: dict) -> DataFrame:
    '''Get player statistics for all players in the given matches'''
    all_players_stats = []
    
    for match_id in matches_info.keys():
        if isinstance(match_id, int):
            # Determine if target team is home or away
            is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam')
            team_key = 'home' if is_home else 'away'
            
            try:
                # Get lineups endpoint
                response = requests.get(api_base + f'events/{match_id}/lineups')
                
                # Check if request was successful
                if response.status_code != 200:
                    print(f"Error fetching player stats for match {match_id}: HTTP {response.status_code}")
                    continue
                
                data = response.json()
                
                # Check if response is valid and has expected structure
                if not isinstance(data, dict):
                    print(f"Error fetching player stats for match {match_id}: Invalid response format")
                    continue
                
                # Get player statistics
                if team_key in data and isinstance(data[team_key], dict) and 'players' in data[team_key]:
                    players = data[team_key]['players']
                    
                    for player in players:
                        if not isinstance(player, dict):
                            continue
                            
                        player_stat = {
                            'match_id': match_id,
                            'player_id': player.get('player', {}).get('id') if isinstance(player.get('player'), dict) else None,
                            'player_name': player.get('player', {}).get('name') if isinstance(player.get('player'), dict) else None,
                            'position': player.get('position'),
                            'shirt_number': player.get('shirtNumber'),
                            'substitute': player.get('substitute', False),
                            'captain': player.get('captain', False)
                        }
                        
                        # Add statistics if available - statistics is a DICT, not a list
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            # Iterate through all statistics in the dictionary
                            for stat_name, stat_value in player['statistics'].items():
                                # Handle nested dictionaries like ratingVersions and statisticsType
                                if isinstance(stat_value, dict):
                                    # For ratingVersions, extract original rating
                                    if stat_name == 'ratingVersions':
                                        player_stat['rating_original'] = stat_value.get('original', 0)
                                        player_stat['rating_alternative'] = stat_value.get('alternative', 0)
                                    # For statisticsType, extract the type
                                    elif stat_name == 'statisticsType':
                                        player_stat['statistics_type'] = stat_value.get('statisticsType', 'player')
                                    # For other nested dicts, skip or convert to string
                                    else:
                                        player_stat[stat_name] = str(stat_value)
                                else:
                                    player_stat[stat_name] = stat_value
                        
                        all_players_stats.append(player_stat)
                else:
                    print(f"No player data found for match {match_id} (team: {team_key})")
                        
            except Exception as e:
                print(f"Error fetching player stats for match {match_id}: {str(e)}")
                continue
    
    if not all_players_stats:
        print("Warning: No player statistics collected")
        return pd.DataFrame()
    
    players_df = pd.DataFrame(all_players_stats).fillna(0)
    
    # Drop the original ratingVersions column if it exists (since we extracted its values)
    if 'ratingVersions' in players_df.columns:
        players_df = players_df.drop('ratingVersions', axis=1)
    
    return players_df

player_stats = get_players_stats(api_base, teams_info)
player_stats.style.background_gradient(cmap='Greens').set_properties(**{'font-family': 'Segoe UI'})
    

In [21]:
player_stats.columns

Index(['match_id', 'player_id', 'player_name', 'position', 'shirt_number',
       'substitute', 'captain', 'totalPass', 'accuratePass', 'totalLongBalls',
       'accurateLongBalls', 'goalAssist', 'accurateOwnHalfPasses',
       'totalOwnHalfPasses', 'accurateOppositionHalfPasses',
       'totalOppositionHalfPasses', 'aerialWon', 'duelWon', 'ballRecovery',
       'goodHighClaim', 'savedShotsFromInsideTheBox', 'saves', 'minutesPlayed',
       'touches', 'rating', 'possessionLostCtrl', 'expectedAssists',
       'keeperSaveValue', 'rating_original', 'rating_alternative',
       'totalShots', 'goalsPrevented', 'passValueNormalized',
       'dribbleValueNormalized', 'defensiveValueNormalized',
       'goalkeeperValueNormalized', 'statistics_type', 'totalCross',
       'aerialLost', 'duelLost', 'challengeLost', 'dispossessed',
       'totalContest', 'shotOffTarget', 'onTargetScoringAttempt',
       'totalClearance', 'outfielderBlock', 'interceptionWon', 'expectedGoals',
       'expectedGoalsO

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">4. | Getting the real position for the player in the field</div>


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">5. | Calculating the score for each player</div>


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">6. | Getting the most recommended players for the match </div>


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">7. | Determining the best plan to play with and suggestion </div>
